In [5]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
NumPy: 2.0.2
Pandas: 2.3.3
Matplotlib: 3.9.4
Scikit-learn: 1.6.1


In [6]:
import pandas as pd

MY_ID = 2729
raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=MY_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print("Формат таблицы:", df.shape)
print("\nТипы данных:")
print(df.dtypes)
print("\nПропущенные значения (NaN):")
print(df.isna().sum())
print("\nСтатистика по столбцам:")
print(df.describe(include="all").T)

Формат таблицы: (1152, 11)

Типы данных:
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object

Пропущенные значения (NaN):
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           58
building_type         0
ceiling_height_m    132
price_kzt             0
dtype: int64

Статистика по столбцам:
                   count unique         top freq         mean        std  min  \
listing_id          1152   1080   ALM-10003    3          NaN        NaN  NaN   
listed_date         1152    246  2026-04-21   11          NaN        NaN  NaN   
district            1152     48       Medeu  142          NaN  

In [7]:
# 1. duplicates
exact_duplicates = df.duplicated().sum()

# 2. reposts
reposts = df.drop_duplicates()["listing_id"].duplicated().sum()

print(f"Полных дубликатов: {exact_duplicates}")
print(f"Перепостов (один listing_id): {reposts}")

Полных дубликатов: 29
Перепостов (один listing_id): 43


### Step 2 Report: Load and Inspect

1. **Column Types & Parsing Issues:**
   - Columns `area_m2`, `price_kzt`, and `rooms` were loaded as `object` instead of numeric types[cite: 1, 2].
   - **Reason:** Raw text contains non-numeric formatting, including thousand-separator spaces, commas instead of decimal dots, symbols (`₸`, `m²`), and text values (treating `"studio"` in `rooms`)[cite: 1, 2].

2. **Data Grain & Duplicates:**
   - **Exact duplicates count:** [29][cite: 1, 2]
   - **Reposts count (duplicate listing_id):** [43][cite: 1, 2]
   - **Data Grain Definition:** One row in the cleaned table represents a single unique apartment listing on its most recent publication date[cite: 1, 2].

3. **Validation Specification:**

| Column | Expected Type | Valid Range / Allowed Values |
| :--- | :--- | :--- |
| `listing_id` | `int64` | Unique listing ID[cite: 1] |
| `listed_date` | `datetime64` | Valid publication date (`YYYY-MM-DD`)[cite: 1] |
| `district` | `category` | Exactly 1 of the 8 Almaty canonical districts[cite: 1] |
| `rooms` | `int64` | Integer >= 1 (`"studio"` = 1)[cite: 1] |
| `area_m2` | `float64` | Total flat area in m² (10 to 500)[cite: 1] |
| `floor` | `int64` | Floor number (1 <= floor <= total_floors)[cite: 1] |
| `total_floors` | `int64` | Building storeys (1 to 50)[cite: 1] |
| `year_built` | `int64` | Construction year (1930 to 2026)[cite: 1] |
| `building_type` | `category` | `panel`, `brick`, or `monolith`[cite: 1] |
| `ceiling_height_m` | `float64` | Ceiling height in meters (2.2 to 5.0)[cite: 1] |
| `price_kzt` | `float64` | Asking price in Tenge (> 0)[cite: 1] |

In [8]:
# Create a copy of the dataset for cleaning
df_clean = df.copy()

# 1. Remove reposts
df_clean = df_clean.drop_duplicates(subset=['listing_id'], keep='first')

# 2. Clean 'rooms' column (convert "studio" to 1 and change type to integer)
df_clean['rooms'] = df_clean['rooms'].astype(str).str.lower().str.replace('studio', '1')
df_clean['rooms'] = df_clean['rooms'].astype(int)

# 3. Clean 'area_m2' column (replace commas with dots, remove letters/symbols, convert to float)
df_clean['area_m2'] = df_clean['area_m2'].astype(str).str.replace(',', '.')
df_clean['area_m2'] = df_clean['area_m2'].str.replace(r'[^\d.]', '', regex=True).astype(float)

# 4. Clean 'price_kzt' column (remove spaces and the ₸ symbol, convert to float)
df_clean['price_kzt'] = df_clean['price_kzt'].astype(str).str.replace(r'[^\d]', '', regex=True).astype(float)

# 5. Apply strict filters based on our Validation Specification
valid_mask = (
        (df_clean['rooms'] >= 1) &
        (df_clean['area_m2'] >= 10) & (df_clean['area_m2'] <= 500) &
        (df_clean['price_kzt'] > 0) &
        (df_clean['total_floors'] >= 1) & (df_clean['total_floors'] <= 50) &
        (df_clean['floor'] >= 1) & (df_clean['floor'] <= df_clean['total_floors']) &
        (df_clean['year_built'] >= 1930) & (df_clean['year_built'] <= 2026) &
        (df_clean['ceiling_height_m'] >= 2.2) & (df_clean['ceiling_height_m'] <= 5.0)
)

# Keep only the rows that pass all validation checks
df_clean = df_clean[valid_mask].reset_index(drop=True)

# Check the results
print("Rows BEFORE cleaning:", len(df))
print("Rows AFTER cleaning:", len(df_clean))
print(f"Dropped rows (errors/duplicates): {len(df) - len(df_clean)}")
print("\nNew data types for cleaned columns:")
print(df_clean[['rooms', 'area_m2', 'price_kzt']].dtypes)

Rows BEFORE cleaning: 1152
Rows AFTER cleaning: 841
Dropped rows (errors/duplicates): 311

New data types for cleaned columns:
rooms          int64
area_m2      float64
price_kzt    float64
dtype: object


In [9]:
# Step 4: Handle Outliers using IQR (Interquartile Range) Method

# 1. Calculate Q1 (25th percentile) and Q3 (75th percentile) for price_kzt
Q1 = df_clean['price_kzt'].quantile(0.25)
Q3 = df_clean['price_kzt'].quantile(0.75)
IQR = Q3 - Q1

# 2. Define upper and lower bound
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Filtering data set to keep prices within bound
df_no_outliers = df_clean[
    (df_clean['price_kzt'] >= lower_bound) &
    (df_clean['price_kzt'] <= upper_bound)
    ].reset_index(drop=True)

# 4. Summary output
outliers_removed = len(df_clean) - len(df_no_outliers)

print("--- Step 4: Outliers Summary ---")
print(f"Lower Bound: {lower_bound:,.0f} ₸")
print(f"Upper Bound: {upper_bound:,.0f} ₸")
print(f"Outliers dropped: {outliers_removed}")
print(f"Rows remaining: {len(df_no_outliers)}")

--- Step 4: Outliers Summary ---
Lower Bound: -16,880,000 ₸
Upper Bound: 102,960,000 ₸
Outliers dropped: 26
Rows remaining: 815
